# APT API Integration Test

Tests the full prod-mode pipeline: create streams, configure, send snapshots, verify pipeline runs.

**Prerequisites:**
1. Start the server in prod mode: `./start.sh --prod`
2. The UI should be blank until snapshots are sent via this notebook.

In [1]:
import httpx
from datetime import datetime, timedelta, timezone
from pprint import pprint



BASE = "localhost:8000"
BASE = "apt-admin.up.railway.app"
client = httpx.Client(base_url=f"https://{BASE}/", timeout=30.0)

## 1. Health Check

In [2]:
r = client.get("/api/health")
print(r.status_code, r.json())

200 {'status': 'ok'}


## 2. Verify Empty State (prod mode)

In prod mode, no streams or pipeline data should exist yet.

In [3]:
r = client.get("/api/streams")
print("Streams:", r.json())
assert r.json()["streams"] == [], "Expected no streams in prod mode"

Streams: {'streams': [{'stream_name': 'rv', 'key_cols': ['symbol', 'expiry'], 'status': 'READY', 'scale': 1.0, 'offset': 0.0, 'exponent': 2.0, 'block': {'annualized': True, 'size_type': 'fixed', 'aggregation_logic': 'average', 'temporal_position': 'shifting', 'decay_end_size_mult': 1.0, 'decay_rate_prop_per_min': 0.0, 'decay_profile': 'linear', 'var_fair_ratio': 1.0}}, {'stream_name': 'mean_iv', 'key_cols': ['symbol', 'expiry'], 'status': 'READY', 'scale': 1.0, 'offset': 0.0, 'exponent': 2.0, 'block': {'annualized': True, 'size_type': 'relative', 'aggregation_logic': 'offset', 'temporal_position': 'shifting', 'decay_end_size_mult': 0.0, 'decay_rate_prop_per_min': 0.01, 'decay_profile': 'linear', 'var_fair_ratio': 2.0}}, {'stream_name': 'events', 'key_cols': ['symbol', 'expiry', 'event_id'], 'status': 'READY', 'scale': 0.01, 'offset': 0.0, 'exponent': 2.0, 'block': {'annualized': False, 'size_type': 'fixed', 'aggregation_logic': 'offset', 'temporal_position': 'static', 'decay_end_size_m

AssertionError: Expected no streams in prod mode

## 3. Timestamps

Use real wall-clock time for `NOW` and a future expiry.

In [ ]:
NOW = datetime.now(timezone.utc).replace(tzinfo=None)
NOW_ISO = NOW.isoformat()
SYMBOL = "BTC"
EXPIRY = NOW + timedelta(days=1)
EXPIRY_ISO = EXPIRY.isoformat()

print(f"NOW:    {NOW_ISO}")
print(f"EXPIRY: {EXPIRY_ISO}")

NOW:    2026-03-22T14:21:21.719446
EXPIRY: 2026-03-23T14:21:21.719446


## 4. Create Streams

Mirror the mock scenario: `rv`, `mean_iv`, `events`.

In [ ]:
STREAMS = [
    {"stream_name": "rv",      "key_cols": ["symbol", "expiry"]},
    {"stream_name": "mean_iv", "key_cols": ["symbol", "expiry"]},
    {"stream_name": "events",  "key_cols": ["symbol", "expiry", "event_id"]},
]

for s in STREAMS:
    r = client.post("/api/streams", json=s)
    print(f"{s['stream_name']}: {r.status_code}")
    pprint(r.json())

rv: 201
{'block': None,
 'exponent': None,
 'key_cols': ['symbol', 'expiry'],
 'offset': None,
 'scale': None,
 'status': 'PENDING',
 'stream_name': 'rv'}
mean_iv: 201
{'block': None,
 'exponent': None,
 'key_cols': ['symbol', 'expiry'],
 'offset': None,
 'scale': None,
 'status': 'PENDING',
 'stream_name': 'mean_iv'}
events: 201
{'block': None,
 'exponent': None,
 'key_cols': ['symbol', 'expiry', 'event_id'],
 'offset': None,
 'scale': None,
 'status': 'PENDING',
 'stream_name': 'events'}


## 5. Configure Streams (admin)

Set pipeline-facing parameters (scale, offset, exponent, block config) to move each stream to READY.

In [ ]:
CONFIGS = {
    "rv": {
        "scale": 1.0,
        "offset": 0.0,
        "exponent": 2,
        "block": {"annualized": True},
    },
    "mean_iv": {
        "scale": 1.0,
        "offset": 0.0,
        "exponent": 2,
        "block": {
            "annualized": True,
            "aggregation_logic": "offset",
            "size_type": "relative",
            "decay_end_size_mult": 0.0,
            "decay_rate_prop_per_min": 0.01,
            "var_fair_ratio": 2.0,
        },
    },
    "events": {
        "scale": 0.01,
        "offset": 0.0,
        "exponent": 2,
        "block": {
            "annualized": False,
            "aggregation_logic": "offset",
            "temporal_position": "static",
            "decay_end_size_mult": 0.0,
            "decay_rate_prop_per_min": 0.01,
            "var_fair_ratio": 3.0,
        },
    },
}

for name, cfg in CONFIGS.items():
    r = client.post(f"/api/streams/{name}/configure", json=cfg)
    print(f"{name}: {r.status_code} -> status={r.json().get('status')}")

rv: 200 -> status=READY
mean_iv: 200 -> status=READY
events: 200 -> status=READY


## 6. Set Bankroll

In [ ]:
r = client.patch("/api/config/bankroll", json={"bankroll": 100_000})
print(r.status_code, r.json())

200 {'bankroll': 100000.0, 'pipeline_rerun': False}


## 7. Set Market Pricing

Space IDs are derived from event `start_timestamp` values by the pipeline.
Format: `static_{YYYYMMDD}_{HHMMSS}` for static blocks, `shifting` for shifting blocks.

In [ ]:
NUM_EVENTS = 5
EVENT_STARTS = [NOW + timedelta(hours=i * 4) for i in range(NUM_EVENTS)]

# Build market pricing keys matching the static space IDs the pipeline will generate
MARKET_PRICING = {"shifting": 0.55}
for ts in EVENT_STARTS:
    space_id = f"static_{ts.strftime('%Y%m%d_%H%M%S')}"
    MARKET_PRICING[space_id] = round(0.20 + 0.10 * EVENT_STARTS.index(ts), 2)

print("Market pricing keys:")
for k, v in MARKET_PRICING.items():
    print(f"  {k}: {v}")

r = client.post("/api/market-pricing", json={"pricing": MARKET_PRICING})
print(f"\nResponse: {r.status_code}", r.json())

Market pricing keys:
  shifting: 0.55
  static_20260322_142121: 0.2
  static_20260322_182121: 0.3
  static_20260322_222121: 0.4
  static_20260323_022121: 0.5
  static_20260323_062121: 0.6

Response: 200 {'spaces_updated': 6, 'pipeline_rerun': False}


## 8. Send Snapshots

Send snapshot data for all three streams. This triggers `rerun_pipeline()` and the WS ticker restarts — the UI should update.

In [ ]:
# -- rv stream --
r = client.post("/api/snapshots", json={
    "stream_name": "rv",
    "rows": [
        {"timestamp": NOW_ISO, "symbol": SYMBOL, "expiry": EXPIRY_ISO, "raw_value": 0.45},
    ],
})
print(f"rv: {r.status_code}", r.json())

# -- mean_iv stream --
r = client.post("/api/snapshots", json={
    "stream_name": "mean_iv",
    "rows": [
        {"timestamp": NOW_ISO, "symbol": SYMBOL, "expiry": EXPIRY_ISO, "raw_value": 0.50},
    ],
})
print(f"mean_iv: {r.status_code}", r.json())

# -- events stream --
event_rows = []
for i in range(NUM_EVENTS):
    event_rows.append({
        "timestamp": NOW_ISO,
        "symbol": SYMBOL,
        "expiry": EXPIRY_ISO,
        "event_id": f"event_{i}",
        "raw_value": [2.5, 3.1, 1.8, 4.0, 2.0][i],
        "start_timestamp": EVENT_STARTS[i].isoformat(),
    })

r = client.post("/api/snapshots", json={
    "stream_name": "events",
    "rows": event_rows,
})
print(f"events: {r.status_code}", r.json())

rv: 200 {'stream_name': 'rv', 'rows_accepted': 1, 'pipeline_rerun': True}
mean_iv: 200 {'stream_name': 'mean_iv', 'rows_accepted': 1, 'pipeline_rerun': True}
events: 200 {'stream_name': 'events', 'rows_accepted': 5, 'pipeline_rerun': True}


## 9. Verify

Check that the pipeline ran and streams are listed as READY with snapshots.

In [ ]:
r = client.get("/api/streams")
for s in r.json()["streams"]:
    print(f"  {s['stream_name']:10s}  status={s['status']}")

print("\nAll streams READY:", all(s["status"] == "READY" for s in r.json()["streams"]))
print("\nOpen the UI at http://localhost:5173 — pipeline data should now be visible.")

  rv          status=READY
  mean_iv     status=READY
  events      status=READY

All streams READY: True

Open the UI at http://localhost:5173 — pipeline data should now be visible.


## 10. (Optional) Update a Snapshot

Send updated values to see the pipeline re-run and the UI refresh.

In [ ]:
# Bump rv raw_value to see a change in the pipeline output
r = client.post("/api/snapshots", json={
    "stream_name": "rv",
    "rows": [
        {"timestamp": NOW_ISO, "symbol": SYMBOL, "expiry": EXPIRY_ISO, "raw_value": 0.55},
    ],
})
print(f"rv update: {r.status_code}", r.json())

rv update: 200 {'stream_name': 'rv', 'rows_accepted': 1, 'pipeline_rerun': True}


## 11. Subscribe to WebSocket — Desired Positions

Connect to the prod engine's `/ws` endpoint and print the desired-position rows as they arrive. Interrupt the cell to stop.

In [ ]:
import asyncio
import json
import websockets

WS_URL = f"wss://{BASE}/ws"
MAX_TICKS = 60  # number of ticks to print before stopping (set to None for unlimited)

async def subscribe_positions():
    async with websockets.connect(WS_URL) as ws:
        print(f"Connected to {WS_URL}\n")
        tick = 0
        async for raw in ws:
            data = json.loads(raw)
            positions = data.get("positions", [])
            ts = data.get("context", {}).get("lastUpdateTimestamp")
            print(f"── Tick {tick} ── (engine ts: {ts})")
            for p in positions:
                print(
                    f"  {p['asset']:>5s}  {p['expiry']:>9s}  "
                    f"desired={p['desiredPos']:>10.2f}  "
                    f"raw={p['rawDesiredPos']:>10.2f}  "
                    f"edge={p['edge']:.6f}  "
                    f"var={p['variance']:.6f}"
                )
            print()
            tick += 1
            if MAX_TICKS is not None and tick >= MAX_TICKS:
                print(f"Stopped after {MAX_TICKS} ticks.")
                break

await subscribe_positions()

Connected to wss://apt-admin.up.railway.app/ws

── Tick 0 ── (engine ts: 1774189625311)
    BTC    23MAR26  desired=  18999.87  raw=  14042.59  edge=0.000003  var=0.000025

── Tick 1 ── (engine ts: 1774189625311)
    BTC    23MAR26  desired=  18999.87  raw=  14042.59  edge=0.000003  var=0.000025

── Tick 2 ── (engine ts: 1774189625311)
    BTC    23MAR26  desired=  18999.87  raw=  14042.59  edge=0.000003  var=0.000025

